# Exercise 4 — Join and aggregate

**Learner exercise** · [All exercises](../index.html) · [Setup](../README.md)

## What you’ll learn

- Choose a join that meets the reporting requirement for sales with no matching product.
- Group sales by category, calculate counts and totals, and check that the report preserves the accepted sales.

**Core: about 10 minutes.** The same baseline for everyone. [Optional zoom-in](#zoom): about 5 extra minutes; choose it here if the topic interests you.

Complete **Your code**, run the **Check** cells, and open hints when needed. Replace `todo(...)` with your answer. Do not use **Run All** while tasks remain unfinished.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../docs/RECOVERY.md).

In [ ]:
import sys
from pathlib import Path

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'lab_support/runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

from lab_support import checks as check
from lab_support.arrival_files import publish_arrival
from lab_support.checks import todo
from lab_support.runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path
from lab_support.workspace import Workspace

workspace = Workspace(solutions=False)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

---
<a id="exercise-4"></a>
## Your task

**What could make this report lose sales or count them twice?**

Core budget: about 10 minutes.

Retain every accepted sale, even without a product match. Label a missing category `unmapped` (our reporting policy). Build one report row per category, with `sales` as a row count and `total` as the sum of amount.

Implement `enrich_sales` and `category_totals`; they will also be used for the stream.

### Supplied — check the lookup first

One category per product key is required. Duplicate lookup keys would multiply sales rows.

In [ ]:
check.lookup(products)

### Your code — enrich the sales

In [ ]:
def enrich_sales(accepted: DataFrame, products: DataFrame) -> DataFrame:
    """Retain sales and add their category where available."""
    return (
        accepted.join(products, on="product_id", how=todo("4: choose the join type"))
        .withColumn("category", F.coalesce("category", F.lit("unmapped")))
        .select("sale_id", "product_id", "amount", "sold_at", "category")
    )

### Your code — the category report

In [ ]:
def category_totals(enriched: DataFrame) -> DataFrame:
    """Produce one row per category with sales and total columns."""
    return enriched.groupBy(todo("4: grouping column")).agg(
        todo("4: row-count expression").alias("sales"),
        todo("4: amount-sum expression").alias("total"),
    )

In [ ]:
enriched = enrich_sales(accepted, products)
report = category_totals(enriched)
report.orderBy("category").show()

### Check

| category | sales | total |
|---|---:|---:|
| books | 3 | 50.00 |
| games | 1 | 40.00 |
| unmapped | 1 | 10.00 |

Five accepted sales must still total 100.00. Explain the `unmapped` row in your own words.

In [ ]:
check.report(enriched, report)

Slide reminders: [Choose which rows to keep](https://dannyscodecorner.github.io/mastering-pyspark/#join-types) · [Group the rows. Calculate for each group.](https://dannyscodecorner.github.io/mastering-pyspark/#grouping-aggregates).

<details>
<summary>Need a nudge? Hint 1</summary>

Which join preserves rows from the left side when a key has no right-hand match?

</details>

<details>
<summary>A little more help: Hint 2</summary>

Use a row count, not a count of nullable category values. `agg` can receive multiple expressions; give them the output names in the task.

</details>

If you need to catch up during class, use the explicit [recovery step](../docs/RECOVERY.md#exercise-4). [Worked solution](../solutions/04-join-aggregate.ipynb) — open it separately when you are ready to compare.

## Core complete

For the 60-minute lab, [skip to Save and finish](#finish). To explore this topic further, continue with the optional section below. Later core exercises do not need any of its variables.

<a id="zoom"></a>
## Optional zoom-in · about 5 minutes

These investigations make up the extra depth in a 90-minute session. Choose them independently; keep your working pipeline unchanged.

### Compare — without changing the working pipeline

Build a separate `inner_sales` DataFrame using an inner join. Inspect its count and total. Which sale was lost? Keep `enriched` and your reusable function unchanged.

In [ ]:
inner_sales = todo("4: a separate inner-join comparison")
inner_sales.agg(F.count("*").alias("sales"), F.sum("amount").alias("total")).show()
check.inner_join(inner_sales)

### Keep track of what one row means

Before aggregation, a row is an accepted sale. After grouping, a row is a category summary. `unmapped` is a reporting choice, not a join keyword. Normalising keys fixes spelling; it does not invent a missing M1 product.

Explore [semi/anti joins and duplicate lookup keys](deeper/join-investigations.ipynb) or [a daily report](deeper/daily-report.ipynb) after the core. Use separate variables for these experiments so the stream keeps its original lookup.

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [ ]:
workspace.save(enrich_sales, category_totals)
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Next: [Exercise 5 — Inspect and save the report](05-save-report.ipynb).

Want more on this topic? You can open these now, using the same saved work: [Investigating joins](deeper/join-investigations.ipynb) · [A daily report](deeper/daily-report.ipynb).

After your attempt, compare the separate [worked solution](../solutions/04-join-aggregate.ipynb).